# 新疆多级分类模型训练
本Notebook实现：
1. 先对大类（Formation）进行分类。
2. 再在每个大类下，训练小类（Alliance/num）分类模型。

In [7]:
# 导入所需库
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 只显示error

## 读取与预处理数据（统一切分训练集和测试集）

In [8]:
# 读取训练数据和类别映射文件
train_data = pd.read_csv(r'F:/TensorFlow/xinjiang/traindata20250626_2.csv')
class_mapping = pd.read_csv(r'F:/TensorFlow/xinjiang/traindata20250626_2_class_mapping.csv')

# 去除Index列
if 'Index' in train_data.columns:
    train_data = train_data.drop(columns=['Index'])

# 合并class_mapping，将Alliance映射为num和Formation
train_data = train_data.merge(class_mapping[['Alliance', 'num', 'Formation']], on='Alliance', how='left')

# 特征、标签准备
feature_cols = [col for col in train_data.columns if col not in ['Alliance', 'num', 'Formation']]
X = train_data[feature_cols]
num_labels = train_data['num']
formation_labels = train_data['Formation']

# 编码大类标签
formation_encoder = LabelEncoder()
formation_y = formation_encoder.fit_transform(formation_labels)

# 统一切分训练集和测试集
X_train, X_test, num_train, num_test, formation_train, formation_test, formation_y_train, formation_y_test = train_test_split(
    X, num_labels, formation_labels, formation_y, test_size=0.2, random_state=42, stratify=num_labels
)

## 构建神经网络与损失函数

In [9]:
def build_model(input_dim, num_classes, use_bn_dropout=True):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Dense(256, activation='relu', input_dim=input_dim, kernel_regularizer=regularizers.l2(0.001)))
    if use_bn_dropout:
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Dropout(0.3))
        
    model.add(tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    if use_bn_dropout:
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Dropout(0.3))
        
    model.add(tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    if use_bn_dropout:
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Dropout(0.3))
        
    model.add(tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    if use_bn_dropout:
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Dropout(0.3))
        
    model.add(tf.keras.layers.Dense(num_classes, activation='softmax'))
    return model

def multi_category_focal_loss2(gamma=2., alpha=.25, class_weights=None):
    epsilon = 1.e-7
    gamma = float(gamma)
    alpha = tf.constant(alpha, dtype=tf.float32)
    if class_weights is not None:
        weights = np.array([class_weights.get(i, 1.0) for i in range(len(class_weights))])
        class_weights_tf = tf.constant(weights, dtype=tf.float32)
    else:
        class_weights_tf = None
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        y_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        ce = -tf.math.log(y_t)
        weight = tf.pow(1. - y_t, gamma)
        fl = alpha_t * weight * ce
        if class_weights_tf is not None:
            fl = fl * class_weights_tf
        return tf.reduce_mean(fl)
    return focal_loss_fixed

def train_and_evaluate(x_train, y_train, num_classes, verbose=0, loss_type='focal', use_bn_dropout=True, show_train_summary=True):
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
    class_weights_array = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
    class_weights = dict(enumerate(class_weights_array))
    
    model = build_model(x_train.shape[1], num_classes, use_bn_dropout=use_bn_dropout)
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
    
    if loss_type == 'focal':
        loss_fn = multi_category_focal_loss2(alpha=0.25, gamma=2, class_weights=class_weights)
    else:
        loss_fn = tf.keras.losses.CategoricalCrossentropy()
        
    model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])
    early_stopping = EarlyStopping(monitor='loss', patience=30, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, verbose=0)
    history = model.fit(x_train, y_train_cat, epochs=500, batch_size=64, verbose=verbose, callbacks=[reduce_lr, early_stopping])
    
    if show_train_summary:
        n_epochs = len(history.history['loss'])
        final_loss = history.history['loss'][-1]
        final_acc = history.history['acc'][-1] if 'acc' in history.history else history.history['accuracy'][-1]
        print(f"训练轮数: {n_epochs}, 最终训练精度: {final_acc:.4f}, 最终训练损失: {final_loss:.4f}")
    
    return model

## 训练分类模型

In [10]:
# 训练大类模型（直接用X_train, formation_y_train）
num_formation_classes = len(np.unique(formation_y_train))
print(f"大类（Formation）类别数: {num_formation_classes}")
formation_model = train_and_evaluate(X_train, formation_y_train, num_formation_classes)
print('大类训练结束')

# 针对每个大类，训练小类（num）分类模型（直接用X_train, num_train, formation_y_train）
formation_list = formation_encoder.classes_
small_class_models = {}
for idx, formation in enumerate(formation_list):
    mask = (formation_y_train == idx)
    X_sub = X_train[mask]
    y_sub = num_train[mask]
    num_encoder = LabelEncoder()
    y_sub_encoded = num_encoder.fit_transform(y_sub)
    n_classes = len(np.unique(y_sub_encoded))
    if n_classes == 1:
        print(f"大类[{formation}] 中只有一个小类，无需训练模型，直接返回该类别")
        continue
    print(f"大类[{formation}]，中的小类数: {n_classes}")
    model = train_and_evaluate(X_sub, y_sub_encoded, n_classes)
    # 可选：保存模型、编码器等
    small_class_models[formation] = (model, num_encoder)
print('小类训练结束')

大类（Formation）类别数: 9
训练轮数: 403, 最终训练精度: 0.7356, 最终训练损失: 0.0185
大类训练结束
大类[丛生草类草原]，中的小类数: 7
训练轮数: 500, 最终训练精度: 0.9568, 最终训练损失: 0.0268
大类[丛生草类草甸]，中的小类数: 3
训练轮数: 500, 最终训练精度: 1.0000, 最终训练损失: 0.1665
大类[农业植被] 中只有一个小类，无需训练模型，直接返回该类别
大类[半乔木与灌木荒漠]，中的小类数: 3
训练轮数: 317, 最终训练精度: 0.9355, 最终训练损失: 0.2313
大类[半灌木与草本荒漠]，中的小类数: 12
训练轮数: 384, 最终训练精度: 0.8801, 最终训练损失: 0.0139
大类[半灌木草原] 中只有一个小类，无需训练模型，直接返回该类别
大类[杂类草草原] 中只有一个小类，无需训练模型，直接返回该类别
大类[根茎草类草甸]，中的小类数: 3
训练轮数: 116, 最终训练精度: 0.8322, 最终训练损失: 0.3728
大类[裸地] 中只有一个小类，无需训练模型，直接返回该类别
小类训练结束


### 评估

In [11]:
print("=== 盲猜基准测试 (Majority Class Baseline) ===")
# 大类盲猜：全猜训练集中数量最多的大类
majority_formation = pd.Series(formation_y_train).mode()[0]
formation_baseline_acc = np.mean(formation_y_test == majority_formation)
print(f"大类 (Formation) 盲猜准确率: {formation_baseline_acc:.6f}")

# 小类盲猜：全猜训练集中数量最多的小类
majority_num = pd.Series(num_train).mode()[0]
num_baseline_acc = np.mean(num_test == majority_num)
print(f"小类 (群系) 盲猜准确率: {num_baseline_acc:.6f}")
print("================================================\n")

# 大类分类器对测试集进行预测
formation_pred_prob = formation_model.predict(X_test)
formation_pred = np.argmax(formation_pred_prob, axis=1)

acc = accuracy_score(formation_y_test, formation_pred)
prec = precision_score(formation_y_test, formation_pred, average='weighted', zero_division=0)
rec = recall_score(formation_y_test, formation_pred, average='weighted', zero_division=0)
f1 = f1_score(formation_y_test, formation_pred, average='weighted', zero_division=0)
print(f"植被型: accuracy: {acc:.6f}, precision: {prec:.6f}, recall: {rec:.6f}, f1: {f1:.6f}")

# 基于大类预测结果，对小类进行预测
num_pred = []
for i in range(len(X_test)):
    x_row = X_test.iloc[[i]]
    # 先预测大类
    formation_pred_prob = formation_model.predict(x_row)
    formation_pred_idx = np.argmax(formation_pred_prob, axis=1)[0]
    formation_name = formation_encoder.classes_[formation_pred_idx]
    # 如果该大类下只有一个小类，直接返回该小类
    mask = (formation_y_train == formation_pred_idx)
    y_sub = num_train[mask]
    if len(np.unique(y_sub)) == 1:
        num_pred.append(np.unique(y_sub)[0])
        continue
    # 用对应小类分类器预测
    model, num_encoder = small_class_models[formation_name]
    num_pred_prob = model.predict(x_row)
    num_pred_idx = np.argmax(num_pred_prob, axis=1)[0]
    num_pred_value = num_encoder.inverse_transform([num_pred_idx])[0]
    num_pred.append(num_pred_value)

# 计算num的四项指标
acc = accuracy_score(num_test, num_pred)
prec = precision_score(num_test, num_pred, average='weighted', zero_division=0)
rec = recall_score(num_test, num_pred, average='weighted', zero_division=0)
f1 = f1_score(num_test, num_pred, average='weighted', zero_division=0)
print(f"群系：accuracy: {acc:.6f}, precision: {prec:.6f}, recall: {rec:.6f}, f1: {f1:.6f}")

=== 盲猜基准测试 (Majority Class Baseline) ===
大类 (Formation) 盲猜准确率: 0.265233
小类 (群系) 盲猜准确率: 0.200717

植被型: accuracy: 0.673835, precision: 0.677336, recall: 0.673835, f1: 0.672969
群系：accuracy: 0.491039, precision: 0.501876, recall: 0.491039, f1: 0.485492


### 消融实验

In [16]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import pandas as pd
import numpy as np

# 辅助函数：根据训练集词频划分多数类/少数类，并计算 F1 宏平均和局部 Accuracy
def get_detailed_metrics(y_true, y_pred, y_train):
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    
    _, _, macro_f1, _ = precision_recall_fscore_support(y_true_arr, y_pred_arr, average='macro', zero_division=0)
    
    train_counts = pd.Series(y_train).value_counts()
    median_freq = train_counts.median()
    majority_classes = train_counts[train_counts >= median_freq].index.tolist()
    minority_classes = train_counts[train_counts < median_freq].index.tolist()
    
    labels = np.unique(y_true_arr)
    _, _, f1_per_class, _ = precision_recall_fscore_support(y_true_arr, y_pred_arr, labels=labels, average=None, zero_division=0)
    f1_dict = dict(zip(labels, f1_per_class))
    
    maj_f1s = [f1_dict[c] for c in majority_classes if c in f1_dict]
    min_f1s = [f1_dict[c] for c in minority_classes if c in f1_dict]
    maj_f1_avg = np.mean(maj_f1s) if len(maj_f1s) > 0 else 0.0
    min_f1_avg = np.mean(min_f1s) if len(min_f1s) > 0 else 0.0
    
    maj_mask = np.isin(y_true_arr, majority_classes)
    min_mask = np.isin(y_true_arr, minority_classes)
    
    maj_acc = accuracy_score(y_true_arr[maj_mask], y_pred_arr[maj_mask]) if maj_mask.sum() > 0 else 0.0
    min_acc = accuracy_score(y_true_arr[min_mask], y_pred_arr[min_mask]) if min_mask.sum() > 0 else 0.0
    
    return macro_f1, maj_f1_avg, min_f1_avg, maj_acc, min_acc

baseline_form_acc = accuracy_score(formation_y_test, formation_pred)
baseline_form_f1 = f1_score(formation_y_test, formation_pred, average='weighted', zero_division=0)
baseline_num_acc = accuracy_score(num_test, num_pred)
baseline_num_f1 = f1_score(num_test, num_pred, average='weighted', zero_division=0)
baseline_macro_f1, baseline_maj_f1, baseline_min_f1, baseline_maj_acc, baseline_min_acc = get_detailed_metrics(num_test, num_pred, num_train)

# ================= 消融实验 (Ablation Studies) =================
print("\n" + "="*60)
print("=== 开始消融实验 (对比评估) ===")
print("="*60)

# 【消融实验 1】不使用先大类后小类的分级训练 (直接扁平化预测小类)
print("\n--- 消融实验 1: 扁平化训练 vs 分级训练 ---")
num_encoder_flat = LabelEncoder()
num_y_train_flat = num_encoder_flat.fit_transform(num_train)
num_classes_flat = len(num_encoder_flat.classes_)

flat_model = train_and_evaluate(X_train, num_y_train_flat, num_classes_flat, show_train_summary=False)

flat_pred_prob = flat_model.predict(X_test)
flat_pred_idx = np.argmax(flat_pred_prob, axis=1)
flat_pred_num_str = num_encoder_flat.inverse_transform(flat_pred_idx)

acc_flat_num = accuracy_score(num_test, flat_pred_num_str)
f1_flat_num = f1_score(num_test, flat_pred_num_str, average='weighted', zero_division=0)

num_to_formation = dict(zip(train_data['num'], train_data['Formation']))
flat_pred_form_str = [num_to_formation.get(n, "Unknown") for n in flat_pred_num_str]
flat_pred_form_enc = formation_encoder.transform(flat_pred_form_str)

acc_flat_form = accuracy_score(formation_y_test, flat_pred_form_enc)
f1_flat_form = f1_score(formation_y_test, flat_pred_form_enc, average='weighted', zero_division=0)

print("[基准模型 (分级训练)]")
print(f"  大类评估 -> accuracy: {baseline_form_acc:.6f}, f1: {baseline_form_f1:.6f}")
print(f"  小类评估 -> accuracy: {baseline_num_acc:.6f}, f1: {baseline_num_f1:.6f}")
print("[消融模型 (无分级，扁平化训练)]")
print(f"  大类评估 -> accuracy: {acc_flat_form:.6f}, f1: {f1_flat_form:.6f}")
print(f"  小类评估 -> accuracy: {acc_flat_num:.6f}, f1: {f1_flat_num:.6f}")


# 包装分级运行逻辑，新增：返回训练好的大类和小类模型，供后续保存
def run_ablation_hierarchical(loss_type='focal', use_bn_dropout=True):
    model_formation_ablation = train_and_evaluate(X_train, formation_y_train, num_formation_classes, loss_type=loss_type, use_bn_dropout=use_bn_dropout, show_train_summary=False)
    
    models_small_ablation = {}
    for idx, formation in enumerate(formation_list):
        mask = (formation_y_train == idx)
        X_sub, y_sub = X_train[mask], num_train[mask]
        sub_encoder = LabelEncoder()
        y_sub_encoded = sub_encoder.fit_transform(y_sub)
        n_classes = len(np.unique(y_sub_encoded))
        if n_classes == 1: continue
        model_s = train_and_evaluate(X_sub, y_sub_encoded, n_classes, loss_type=loss_type, use_bn_dropout=use_bn_dropout, show_train_summary=False)
        models_small_ablation[formation] = (model_s, sub_encoder)

    pred_nums_ablation = []
    for i in range(len(X_test)):
        x_row = X_test.iloc[[i]]
        f_prob = model_formation_ablation.predict(x_row)
        f_idx = np.argmax(f_prob, axis=1)[0]
        f_name = formation_encoder.classes_[f_idx]
        
        mask = (formation_y_train == f_idx)
        y_sub_train = num_train[mask]
        if len(np.unique(y_sub_train)) == 1:
            pred_nums_ablation.append(np.unique(y_sub_train)[0])
            continue

        m, enc = models_small_ablation[f_name]
        s_prob = m.predict(x_row)
        s_idx = np.argmax(s_prob, axis=1)[0]
        pred_nums_ablation.append(enc.inverse_transform([s_idx])[0])

    acc_abl = accuracy_score(num_test, pred_nums_ablation)
    f1_abl = f1_score(num_test, pred_nums_ablation, average='weighted', zero_division=0)
    
    # 修改处：返回大类模型与小类模型字典
    return acc_abl, f1_abl, pred_nums_ablation, model_formation_ablation, models_small_ablation


# 【消融实验 2】替换 Focal loss 为 CrossEntropy
print("\n--- 消融实验 2: Categorical CrossEntropy vs Focal Loss ---")
acc_ce, f1_ce, pred_ce, ce_form_model, ce_small_models = run_ablation_hierarchical(loss_type='ce', use_bn_dropout=True)
ce_macro_f1, ce_maj_f1, ce_min_f1, ce_maj_acc, ce_min_acc = get_detailed_metrics(num_test, pred_ce, num_train)

print("[基准模型 (使用 Focal Loss)]")
print(f"  总体评估 -> accuracy: {baseline_num_acc:.6f}, f1: {baseline_num_f1:.6f}")
print(f"  多数类样本 -> accuracy: {baseline_maj_acc:.6f}, F1平均: {baseline_maj_f1:.6f}")
print(f"  少数类样本 -> accuracy: {baseline_min_acc:.6f}, F1平均: {baseline_min_f1:.6f}")
print(f"  宏平均 F1 (Macro-F1): {baseline_macro_f1:.6f}")

print("[消融模型 (使用 CrossEntropy Loss)]")
print(f"  总体评估 -> accuracy: {acc_ce:.6f}, f1: {f1_ce:.6f}")
print(f"  多数类样本 -> accuracy: {ce_maj_acc:.6f}, F1平均: {ce_maj_f1:.6f}")
print(f"  少数类样本 -> accuracy: {ce_min_acc:.6f}, F1平均: {ce_min_f1:.6f}")
print(f"  宏平均 F1 (Macro-F1): {ce_macro_f1:.6f}")


# 【消融实验 3】移除 BN & Dropout
print("\n--- 消融实验 3: 无 BN & Dropout vs 包含 BN & Dropout ---")
acc_nobn, f1_nobn, _, nobn_form_model, nobn_small_models = run_ablation_hierarchical(loss_type='focal', use_bn_dropout=False)

print("[基准模型 (包含 BatchNormalization 和 Dropout)]")
print(f"  小类总体评估 -> accuracy: {baseline_num_acc:.6f}, f1: {baseline_num_f1:.6f}")
print("[消融模型 (移除 BatchNormalization 和 Dropout)]")
print(f"  小类总体评估 -> accuracy: {acc_nobn:.6f}, f1: {f1_nobn:.6f}")


=== 开始消融实验 (对比评估) ===

--- 消融实验 1: 扁平化训练 vs 分级训练 ---
[基准模型 (分级训练)]
  大类评估 -> accuracy: 0.673835, f1: 0.672969
  小类评估 -> accuracy: 0.491039, f1: 0.485492
[消融模型 (无分级，扁平化训练)]
  大类评估 -> accuracy: 0.670251, f1: 0.668657
  小类评估 -> accuracy: 0.498208, f1: 0.500165

--- 消融实验 2: Categorical CrossEntropy vs Focal Loss ---
[基准模型 (使用 Focal Loss)]
  总体评估 -> accuracy: 0.491039, f1: 0.485492
  多数类样本 -> accuracy: 0.526316, F1平均: 0.388049
  少数类样本 -> accuracy: 0.333333, F1平均: 0.361337
  宏平均 F1 (Macro-F1): 0.375528
[消融模型 (使用 CrossEntropy Loss)]
  总体评估 -> accuracy: 0.476703, f1: 0.482676
  多数类样本 -> accuracy: 0.508772, F1平均: 0.405925
  少数类样本 -> accuracy: 0.333333, F1平均: 0.374392
  宏平均 F1 (Macro-F1): 0.391144

--- 消融实验 3: 无 BN & Dropout vs 包含 BN & Dropout ---
[基准模型 (包含 BatchNormalization 和 Dropout)]
  小类总体评估 -> accuracy: 0.491039, f1: 0.485492
[消融模型 (移除 BatchNormalization 和 Dropout)]
  小类总体评估 -> accuracy: 0.222222, f1: 0.131831


### 保存

In [17]:
import os
import pickle
import datetime

# ================= 新增：保存数据集与模型 =================
print("=== 开始保存数据集与模型 ===")

current_time_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = f"saved_workspace_{current_time_str}"
os.makedirs(save_dir, exist_ok=True)

# 1. 保存切分的数据集
print(f"正在保存训练集与测试集到: {save_dir} ...")
train_save = X_train.copy()
train_save['num'] = num_train
train_save['Formation'] = formation_train
train_save['Formation_encoded'] = formation_y_train
train_save.to_csv(os.path.join(save_dir, "train_set.csv"), index=False, encoding='utf-8-sig')

test_save = X_test.copy()
test_save['num'] = num_test
test_save['Formation'] = formation_test
test_save['Formation_encoded'] = formation_y_test
test_save.to_csv(os.path.join(save_dir, "test_set.csv"), index=False, encoding='utf-8-sig')
print("  -> 数据集保存完成！")

# 提取中英文名称映射（去掉可能的空值或重复）
formation_to_eng = dict(zip(class_mapping['Formation'], class_mapping['Eng_Formation']))

# 定义辅助函数：用于保存分级模型（大类+所有小类），带有灵活的前缀，并使用英文名
def save_hierarchical_models(form_model, small_models, form_encoder, prefix=""):
    # 保存大类模型和编码器
    form_model.save(os.path.join(save_dir, f"{prefix}formation_model.h5"))
    if form_encoder is not None:
        with open(os.path.join(save_dir, f"{prefix}formation_encoder.pkl"), "wb") as f:
            pickle.dump(form_encoder, f)
            
    mapping_info = {}
    for formation_name, (model, encoder) in small_models.items():
        # 获取英文名，如果不存在则退回原名
        eng_name = formation_to_eng.get(formation_name, str(formation_name))
        # 替换特殊字符和空格，确保作为文件名的安全性
        safe_eng_name = str(eng_name).replace("/", "_").replace("\\", "_").replace(" ", "_")
        
        model_filename = f"{prefix}small_model_{safe_eng_name}.h5"
        encoder_filename = f"{prefix}small_encoder_{safe_eng_name}.pkl"
        
        model.save(os.path.join(save_dir, model_filename))
        with open(os.path.join(save_dir, encoder_filename), "wb") as f:
            pickle.dump(encoder, f)
            
        mapping_info[formation_name] = {
            "english_name": eng_name,
            "model_file": model_filename,
            "encoder_file": encoder_filename
        }

    # 将小类文件的配置信息一并保存
    with open(os.path.join(save_dir, f"{prefix}small_class_mapping_info.pkl"), "wb") as f:
        pickle.dump(mapping_info, f)

print(f"正在保存所有模型和编码器到: {save_dir} ...")

# 2. 保存基准模型 (Base)
print("  -> 保存 基准分级模型 (base_)...")
save_hierarchical_models(formation_model, small_class_models, formation_encoder, prefix="base_")

# 3. 保存消融实验1模型 (扁平化)
print("  -> 保存 消融实验1模型 (ablation1_flat_)...")
flat_model.save(os.path.join(save_dir, "ablation1_flat_model.h5"))
with open(os.path.join(save_dir, "ablation1_num_encoder_flat.pkl"), "wb") as f:
    pickle.dump(num_encoder_flat, f)

# 4. 保存消融实验2模型 (CrossEntropy)
print("  -> 保存 消融实验2模型 (ablation2_ce_)...")
# 不重复保存大类编码器，传入 None
save_hierarchical_models(ce_form_model, ce_small_models, None, prefix="ablation2_ce_")

# 5. 保存消融实验3模型 (无BN&Dropout)
print("  -> 保存 消融实验3模型 (ablation3_nobn_)...")
save_hierarchical_models(nobn_form_model, nobn_small_models, None, prefix="ablation3_nobn_")

print("  -> 恭喜！所有数据集和所有对比模型均已保存完成！")
print("="*60)

=== 开始保存数据集与模型 ===
正在保存训练集与测试集到: saved_workspace_20260516_152036 ...
  -> 数据集保存完成！
正在保存所有模型和编码器到: saved_workspace_20260516_152036 ...
  -> 保存 基准分级模型 (base_)...
  -> 保存 消融实验1模型 (ablation1_flat_)...
  -> 保存 消融实验2模型 (ablation2_ce_)...
  -> 保存 消融实验3模型 (ablation3_nobn_)...
  -> 恭喜！所有数据集和所有对比模型均已保存完成！
